# AI-Powered Customer Churn Analysis
### Submitted by: **Sumit Gupta**
### Project: AI-Powered-Customer-Churn-Analysis

---

## Project Overview

This notebook provides a complete, end-to-end walkthrough of building an AI-powered customer churn prediction system. It covers:

1. **Data Loading & Inspection** — Load and audit the raw dataset
2. **Data Cleaning Pipeline** — Handle nulls, duplicates, type coercion
3. **Exploratory Data Analysis (EDA)** — 6+ interactive Plotly visualisations
4. **Business KPIs** — 9+ live key performance indicators
5. **SQL-Style Analysis** — Business queries via Pandas
6. **Feature Engineering & Preprocessing** — StandardScaler + OneHotEncoder
7. **Model Training & Comparison** — Logistic Regression, Random Forest, Gradient Boosting, XGBoost
8. **Model Evaluation** — Confusion Matrix, ROC Curve, Classification Report
9. **Churn Risk Predictions** — Per-customer probability + Low/Medium/High risk tiers
10. **AI-Powered Insights** — LLM-enhanced or rule-based findings & recommendations

---

**Dataset:** AI Customer Churn Dataset (Kaggle / IBM Telco compatible)  
**Dataset Link:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn  
**Tech Stack:** Python, Pandas, NumPy, scikit-learn, XGBoost, Plotly, Streamlit, OpenAI API

## Step 0: Install & Import Libraries

In [ ]:
# ── Install dependencies (uncomment if needed) ────────────────────────────
# !pip install pandas numpy scikit-learn xgboost plotly streamlit joblib python-dotenv openai openpyxl

# ── Standard library imports ──────────────────────────────────────────────
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path so src/ modules are importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Data libraries ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

# ── Machine Learning ─────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

# ── Optional: XGBoost ─────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print('XGBoost available ✓')
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed — skipping (install with: pip install xgboost)')

# ── Project source modules ────────────────────────────────────────────────
try:
    from src.data_cleaning import load_raw_data, clean_data, audit_dataframe, outlier_report
    from src.eda import (
        fig_churn_distribution, fig_tenure_distribution,
        fig_monthly_charges_distribution, fig_contract_distribution,
        fig_correlation_heatmap, fig_churn_by_charges_bin,
        fig_model_comparison, fig_confusion_matrix, fig_roc_curve, fig_feature_importance
    )
    from src.preprocessing import prepare_data
    from src.model_training import train_all_models, select_best_model, detailed_evaluation
    from src.prediction import predict_churn, add_risk_explanations, prediction_summary
    from src.ai_insights import generate_insights
    from src.utils import calculate_kpis, generate_sample_dataset
    print('Project src modules loaded ✓')
except ImportError as e:
    print(f'Note: Could not import src modules ({e}). Using inline implementations below.')

# ── Environment variables (optional — for AI API key) ─────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv('../.env')
    load_dotenv('.env')
except ImportError:
    pass

print('\nAll imports complete.')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')

---
## Step 1: Data Loading & Inspection

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────
# Try multiple possible paths for the dataset
POSSIBLE_PATHS = [
    '../data/raw/ai_customer_churn_dataset.csv',
    '../dataset/ai_customer_churn_dataset.csv',
    'data/raw/ai_customer_churn_dataset.csv',
    '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv',
    'data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv',
]

raw_df = None
for path in POSSIBLE_PATHS:
    if os.path.exists(path):
        raw_df = pd.read_csv(path)
        print(f'✓ Loaded dataset from: {path}')
        print(f'  Shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns')
        break

if raw_df is None:
    print('Dataset not found locally — generating synthetic demo data (n=1000).')
    print('To use the real dataset, download from:')
    print('  https://www.kaggle.com/datasets/blastchar/telco-customer-churn')
    print('  and place in data/raw/')
    try:
        raw_df = generate_sample_dataset(n=1000)
    except Exception:
        # Minimal fallback synthetic data
        import random
        random.seed(42)
        np.random.seed(42)
        n = 1000
        raw_df = pd.DataFrame({
            'customerID': [f'CUST-{i:04d}' for i in range(n)],
            'gender': np.random.choice(['Male', 'Female'], n),
            'SeniorCitizen': np.random.choice([0, 1], n, p=[0.84, 0.16]),
            'Partner': np.random.choice(['Yes', 'No'], n),
            'Dependents': np.random.choice(['Yes', 'No'], n, p=[0.3, 0.7]),
            'tenure': np.random.randint(1, 73, n),
            'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.55, 0.21, 0.24]),
            'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n, p=[0.34, 0.44, 0.22]),
            'MonthlyCharges': np.round(np.random.uniform(18, 118, n), 2),
            'TotalCharges': np.round(np.random.uniform(18, 8500, n), 2),
            'Churn': np.random.choice(['Yes', 'No'], n, p=[0.265, 0.735]),
        })
        print(f'Synthetic demo data generated: {raw_df.shape}')

raw_df.head(10)

In [ ]:
# ── Raw data audit ────────────────────────────────────────────────────────
print('=== RAW DATA AUDIT ===')
print(f'Shape       : {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns')
print(f'Duplicates  : {raw_df.duplicated().sum()}')
print(f'Total nulls : {raw_df.isnull().sum().sum()}')
print()
print('Column types and null counts:')
info_df = pd.DataFrame({
    'Column': raw_df.columns,
    'dtype': raw_df.dtypes.values,
    'Non-Null Count': raw_df.notnull().sum().values,
    'Null Count': raw_df.isnull().sum().values,
    'Null %': (raw_df.isnull().sum().values / len(raw_df) * 100).round(2)
})
print(info_df.to_string(index=False))

In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────
print('=== DESCRIPTIVE STATISTICS (Numeric Columns) ===')
raw_df.describe().T.round(2)

---
## Step 2: Data Cleaning Pipeline

In [ ]:
# ── Run the cleaning pipeline ─────────────────────────────────────────────
try:
    clean_df, report = clean_data(raw_df)
except Exception:
    # Inline cleaning if src module not available
    clean_df = raw_df.copy()
    clean_df.columns = [c.strip() for c in clean_df.columns]
    clean_df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    n_dupes = clean_df.duplicated().sum()
    clean_df = clean_df.drop_duplicates()
    if 'TotalCharges' in clean_df.columns:
        clean_df['TotalCharges'] = pd.to_numeric(clean_df['TotalCharges'], errors='coerce')
        clean_df['TotalCharges'].fillna(clean_df['TotalCharges'].median(), inplace=True)
    if 'Churn' in clean_df.columns:
        if clean_df['Churn'].dtype == object:
            clean_df['Churn'] = clean_df['Churn'].str.strip().map({'Yes': 1, 'No': 0}).fillna(0).astype(int)
    report = {
        'rows_before': len(raw_df), 'rows_after': len(clean_df),
        'nulls_before': raw_df.isnull().sum().sum(), 'nulls_after': clean_df.isnull().sum().sum(),
        'duplicates_removed': int(n_dupes)
    }

print('=== CLEANING SUMMARY ===')
print(f"Rows     : {report['rows_before']:,} → {report['rows_after']:,}")
print(f"Nulls    : {report['nulls_before']} → {report['nulls_after']}")
print(f"Dupes rm : {report['duplicates_removed']}")
print()
print(f'Clean dataset shape: {clean_df.shape}')
clean_df.head()

In [ ]:
# ── Outlier report (IQR method — non-destructive) ─────────────────────────
print('=== OUTLIER REPORT (IQR Method) ===')
print('Note: Outliers are retained — extreme customers (high spend / long tenure) are valuable.')
print()
for col in clean_df.select_dtypes(include=[np.number]).columns:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = int(((clean_df[col] < lower) | (clean_df[col] > upper)).sum())
    pct = round(100 * n_out / len(clean_df), 2)
    print(f'  {col:<25} Q1={q1:.1f}  Q3={q3:.1f}  Outliers={n_out} ({pct}%)')

---
## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1 Churn Distribution ────────────────────────────────────────────────
try:
    fig_churn_distribution(clean_df).show()
except Exception:
    churn_counts = clean_df['Churn'].value_counts().rename({0: 'Active', 1: 'Churned'})
    fig = px.pie(values=churn_counts.values, names=churn_counts.index,
                 title='Churn Distribution',
                 color_discrete_map={'Active': '#2196F3', 'Churned': '#F44336'})
    fig.show()

total = len(clean_df)
churned = clean_df['Churn'].sum()
print(f'Total customers : {total:,}')
print(f'Churned         : {churned:,} ({churned/total*100:.1f}%)')
print(f'Active          : {total-churned:,} ({(total-churned)/total*100:.1f}%)')

In [ ]:
# ── 3.2 Tenure Distribution by Churn ─────────────────────────────────────
try:
    fig_tenure_distribution(clean_df).show()
except Exception:
    if 'tenure' in clean_df.columns:
        fig = px.histogram(clean_df, x='tenure', color='Churn',
                           title='Tenure Distribution by Churn',
                           labels={'Churn': 'Churn (1=Yes, 0=No)'},
                           nbins=50, barmode='overlay', opacity=0.7)
        fig.show()

In [ ]:
# ── 3.3 Monthly Charges Distribution ─────────────────────────────────────
try:
    fig_monthly_charges_distribution(clean_df).show()
except Exception:
    if 'MonthlyCharges' in clean_df.columns:
        fig = px.histogram(clean_df, x='MonthlyCharges', color='Churn',
                           title='Monthly Charges Distribution by Churn',
                           nbins=50, barmode='overlay', opacity=0.7)
        fig.show()

In [ ]:
# ── 3.4 Churn by Contract Type ────────────────────────────────────────────
try:
    fig_contract_distribution(clean_df).show()
except Exception:
    if 'Contract' in clean_df.columns:
        contract_churn = clean_df.groupby('Contract')['Churn'].agg(['sum', 'count']).reset_index()
        contract_churn['churn_rate'] = contract_churn['sum'] / contract_churn['count'] * 100
        fig = px.bar(contract_churn, x='Contract', y='churn_rate',
                     title='Churn Rate by Contract Type (%)',
                     color='churn_rate', color_continuous_scale='Reds')
        fig.show()

In [ ]:
# ── 3.5 Correlation Heatmap ───────────────────────────────────────────────
try:
    fig_correlation_heatmap(clean_df).show()
except Exception:
    num_cols = clean_df.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) > 1:
        corr = clean_df[num_cols].corr()
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                    center=0, ax=ax, linewidths=0.5)
        ax.set_title('Correlation Heatmap')
        plt.tight_layout()
        plt.show()

In [ ]:
# ── 3.6 Churn by Monthly Charges Bin ─────────────────────────────────────
try:
    fig_churn_by_charges_bin(clean_df).show()
except Exception:
    if 'MonthlyCharges' in clean_df.columns:
        df_temp = clean_df.copy()
        df_temp['ChargesBin'] = pd.cut(df_temp['MonthlyCharges'], bins=5,
                                       labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
        bin_churn = df_temp.groupby('ChargesBin', observed=True)['Churn'].mean() * 100
        fig = px.bar(x=bin_churn.index.astype(str), y=bin_churn.values,
                     title='Churn Rate by Monthly Charges Band (%)',
                     labels={'x': 'Charges Band', 'y': 'Churn Rate (%)'},
                     color=bin_churn.values, color_continuous_scale='Reds')
        fig.show()

---
## Step 4: Business KPIs

In [ ]:
# ── Business KPIs ─────────────────────────────────────────────────────────
print('=== KEY PERFORMANCE INDICATORS ===')
print()

total_customers = len(clean_df)
churned_customers = clean_df['Churn'].sum()
churn_rate = churned_customers / total_customers * 100

print(f'  Total Customers       : {total_customers:,}')
print(f'  Churned Customers     : {churned_customers:,}')
print(f'  Churn Rate            : {churn_rate:.2f}%')
print(f'  Retention Rate        : {100 - churn_rate:.2f}%')

if 'MonthlyCharges' in clean_df.columns:
    avg_monthly = clean_df['MonthlyCharges'].mean()
    churned_monthly = clean_df[clean_df['Churn']==1]['MonthlyCharges'].mean()
    monthly_revenue_at_risk = clean_df[clean_df['Churn']==1]['MonthlyCharges'].sum()
    print(f'  Avg Monthly Charges   : ${avg_monthly:.2f}')
    print(f'  Churners Avg Monthly  : ${churned_monthly:.2f}')
    print(f'  Monthly Revenue @ Risk: ${monthly_revenue_at_risk:,.2f}')

if 'TotalCharges' in clean_df.columns:
    total_revenue_lost = clean_df[clean_df['Churn']==1]['TotalCharges'].sum()
    print(f'  Total Revenue Lost    : ${total_revenue_lost:,.2f}')

if 'tenure' in clean_df.columns:
    avg_tenure_all = clean_df['tenure'].mean()
    avg_tenure_churned = clean_df[clean_df['Churn']==1]['tenure'].mean()
    print(f'  Avg Tenure (All)      : {avg_tenure_all:.1f} months')
    print(f'  Avg Tenure (Churned)  : {avg_tenure_churned:.1f} months')

---
## Step 5: SQL-Style Analysis (via Pandas)

In [ ]:
# ── Query 1: Churn by Contract Type ──────────────────────────────────────
print('--- Query 1: Churn Rate by Contract Type ---')
if 'Contract' in clean_df.columns:
    q1 = clean_df.groupby('Contract').agg(
        total_customers=('Churn', 'count'),
        churned=('Churn', 'sum')
    ).reset_index()
    q1['churn_rate_%'] = (q1['churned'] / q1['total_customers'] * 100).round(2)
    q1 = q1.sort_values('churn_rate_%', ascending=False)
    print(q1.to_string(index=False))

In [ ]:
# ── Query 2: Churn by Internet Service ───────────────────────────────────
print('--- Query 2: Churn Rate by Internet Service ---')
if 'InternetService' in clean_df.columns:
    q2 = clean_df.groupby('InternetService').agg(
        total=('Churn', 'count'),
        churned=('Churn', 'sum')
    ).reset_index()
    q2['churn_rate_%'] = (q2['churned'] / q2['total'] * 100).round(2)
    print(q2.sort_values('churn_rate_%', ascending=False).to_string(index=False))

In [ ]:
# ── Query 3: Revenue from churned customers ───────────────────────────────
print('--- Query 3: Revenue Analysis — Churned vs Active ---')
if 'TotalCharges' in clean_df.columns:
    q3 = clean_df.groupby('Churn').agg(
        count=('TotalCharges', 'count'),
        total_revenue=('TotalCharges', 'sum'),
        avg_revenue=('TotalCharges', 'mean')
    ).reset_index()
    q3['Churn'] = q3['Churn'].map({0: 'Active', 1: 'Churned'})
    q3['total_revenue'] = q3['total_revenue'].round(2)
    q3['avg_revenue'] = q3['avg_revenue'].round(2)
    print(q3.to_string(index=False))

In [ ]:
# ── Query 4: Tenure segments and churn risk ───────────────────────────────
print('--- Query 4: Churn Rate by Tenure Segment ---')
if 'tenure' in clean_df.columns:
    df_temp = clean_df.copy()
    df_temp['tenure_segment'] = pd.cut(
        df_temp['tenure'],
        bins=[0, 12, 24, 36, 60, 100],
        labels=['0-12 mo', '13-24 mo', '25-36 mo', '37-60 mo', '60+ mo']
    )
    q4 = df_temp.groupby('tenure_segment', observed=True).agg(
        count=('Churn', 'count'),
        churned=('Churn', 'sum')
    ).reset_index()
    q4['churn_rate_%'] = (q4['churned'] / q4['count'] * 100).round(2)
    print(q4.to_string(index=False))

---
## Step 6: Feature Engineering & Preprocessing

In [ ]:
# ── Prepare data for ML ───────────────────────────────────────────────────
try:
    prep = prepare_data(clean_df)
    X_train = prep['X_train']
    X_test  = prep['X_test']
    y_train = prep['y_train']
    y_test  = prep['y_test']
    preprocessor     = prep['preprocessor']
    feature_cols     = prep['feature_cols']
    feature_names    = prep['feature_names_transformed']
    print('Preprocessing via src.preprocessing ✓')
except Exception:
    # Inline preprocessing fallback
    NON_FEATURE = {'customerid', 'customerID', 'churn', 'churned'}
    feature_cols = [c for c in clean_df.columns if c.lower() not in NON_FEATURE]
    target_col = 'Churn'
    X = clean_df[feature_cols]
    y = clean_df[target_col].astype(int).values
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
    cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                         ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    transformers = []
    if num_cols: transformers.append(('num', num_pipe, num_cols))
    if cat_cols: transformers.append(('cat', cat_pipe, cat_cols))
    preprocessor = ColumnTransformer(transformers, remainder='drop')
    X_train = preprocessor.fit_transform(X_train_raw)
    X_test  = preprocessor.transform(X_test_raw)
    feature_names = num_cols
    if cat_cols:
        ohe = preprocessor.named_transformers_['cat'].named_steps['ohe']
        feature_names += list(ohe.get_feature_names_out(cat_cols))
    print('Inline preprocessing fallback ✓')

print(f'Training samples : {X_train.shape[0]:,}')
print(f'Test samples     : {X_test.shape[0]:,}')
print(f'Features         : {X_train.shape[1]}')
print(f'Churn rate train : {y_train.mean()*100:.2f}%')
print(f'Churn rate test  : {y_test.mean()*100:.2f}%')

---
## Step 7: Model Training & Comparison

In [ ]:
# ── Train all models ──────────────────────────────────────────────────────
print('Training models... (this may take ~30-60 seconds)')

models_dict = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
}
if XGBOOST_AVAILABLE:
    models_dict['XGBoost'] = XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        eval_metric='logloss', random_state=42, verbosity=0
    )

results = []
trained_models = {}

for name, model in models_dict.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else y_pred.astype(float)
    results.append({
        'Model': name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1 Score':  round(f1_score(y_test, y_pred, zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4),
    })
    trained_models[name] = model
    print(f'  ✓ {name}')

results_df = pd.DataFrame(results)[['Model','Accuracy','Precision','Recall','F1 Score','ROC-AUC']]
results_df = results_df.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print('\n=== MODEL COMPARISON ===')
results_df

In [ ]:
# ── Model comparison bar chart ────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
fig = go.Figure()
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
for i, row in results_df.iterrows():
    fig.add_trace(go.Bar(
        name=row['Model'],
        x=metrics,
        y=[row[m] for m in metrics],
        marker_color=colors[i % len(colors)]
    ))
fig.update_layout(
    title='Model Performance Comparison',
    barmode='group',
    yaxis=dict(range=[0, 1.05]),
    height=450
)
fig.show()

---
## Step 8: Best Model Evaluation

In [ ]:
# ── Select and evaluate best model ───────────────────────────────────────
best_name = results_df.iloc[0]['Model']
best_model = trained_models[best_name]
print(f'Best Model: {best_name}')
print(f'ROC-AUC   : {results_df.iloc[0]["ROC-AUC"]}')

y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print('\n=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred_best, target_names=['Active (0)', 'Churned (1)']))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_best)
fig_cm = px.imshow(
    cm,
    labels=dict(x='Predicted', y='Actual', color='Count'),
    x=['Active (0)', 'Churned (1)'],
    y=['Active (0)', 'Churned (1)'],
    title=f'Confusion Matrix — {best_name}',
    color_continuous_scale='Blues',
    text_auto=True
)
fig_cm.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (Active correctly identified) : {tn}')
print(f'False Positives (Active incorrectly flagged)   : {fp}')
print(f'False Negatives (Churners missed)              : {fn}')
print(f'True Positives  (Churners correctly caught)    : {tp}')

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob_best)
auc_score = roc_auc_score(y_test, y_prob_best)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                              name=f'{best_name} (AUC={auc_score:.4f})',
                              line=dict(color='#2196F3', width=2)))
fig_roc.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                              name='Random Classifier',
                              line=dict(color='gray', dash='dash')))
fig_roc.update_layout(
    title='ROC Curve',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=450
)
fig_roc.show()

In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_[0])
else:
    importances = None

if importances is not None and len(feature_names) == len(importances):
    fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    fi_df = fi_df.sort_values('Importance', ascending=False).head(20)
    fig_fi = px.bar(
        fi_df, x='Importance', y='Feature', orientation='h',
        title=f'Top 20 Feature Importances — {best_name}',
        color='Importance', color_continuous_scale='Blues'
    )
    fig_fi.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
    fig_fi.show()
else:
    print('Feature importance not available or feature name count mismatch.')

---
## Step 9: Churn Risk Predictions

In [ ]:
# ── Score all customers ───────────────────────────────────────────────────
try:
    pred_df = predict_churn(clean_df, best_model, preprocessor, feature_cols)
    pred_df = add_risk_explanations(pred_df)
    print('Predictions via src.prediction ✓')
except Exception:
    # Inline prediction fallback
    pred_df = clean_df.copy()
    X_all = preprocessor.transform(clean_df[feature_cols])
    pred_df['Churn Probability'] = (best_model.predict_proba(X_all)[:, 1] * 100).round(2)
    pred_df['Risk Level'] = pd.cut(
        pred_df['Churn Probability'],
        bins=[-1, 33, 66, 101],
        labels=['Low Risk', 'Medium Risk', 'High Risk']
    )

print('=== PREDICTION SUMMARY ===')
total = len(pred_df)
for level in ['High Risk', 'Medium Risk', 'Low Risk']:
    if 'Risk Level' in pred_df.columns:
        n = (pred_df['Risk Level'] == level).sum()
        print(f'  {level:<15}: {n:,} ({n/total*100:.1f}%)')

In [ ]:
# ── View top high-risk customers ──────────────────────────────────────────
print('=== TOP 15 HIGH-RISK CUSTOMERS ===')
id_col = next((c for c in pred_df.columns if 'id' in c.lower() and 'customer' in c.lower()), None)
if id_col is None:
    id_col = next((c for c in pred_df.columns if 'id' in c.lower()), None)

display_cols = []
if id_col: display_cols.append(id_col)
for c in ['Contract', 'tenure', 'MonthlyCharges', 'InternetService',
          'Churn Probability', 'Risk Level']:
    if c in pred_df.columns:
        display_cols.append(c)

if 'Risk Level' in pred_df.columns:
    high_risk = pred_df[pred_df['Risk Level'] == 'High Risk']
    if 'Churn Probability' in high_risk.columns:
        high_risk = high_risk.sort_values('Churn Probability', ascending=False)
    print(f'Total High-Risk customers: {len(high_risk):,}')
    high_risk[display_cols].head(15)

In [ ]:
# ── Risk Level Distribution Chart ────────────────────────────────────────
if 'Risk Level' in pred_df.columns:
    risk_counts = pred_df['Risk Level'].value_counts().reset_index()
    risk_counts.columns = ['Risk Level', 'Count']
    color_map = {'High Risk': '#F44336', 'Medium Risk': '#FF9800', 'Low Risk': '#4CAF50'}
    fig_risk = px.pie(
        risk_counts, values='Count', names='Risk Level',
        title='Customer Risk Distribution',
        color='Risk Level', color_discrete_map=color_map
    )
    fig_risk.show()

---
## Step 10: AI-Powered Insights

In [ ]:
# ── Generate AI/Rule-based insights ──────────────────────────────────────
try:
    insights = generate_insights(clean_df, pred_df)
    print(f'Insight mode : {insights["mode"]}')
    print()
    print('=== KEY FINDINGS ===')
    print(insights['key_findings'])
    print()
    print('=== CHURN DRIVERS ===')
    print(insights['churn_drivers'])
    print()
    print('=== RECOMMENDATIONS ===')
    print(insights['recommendations'])
except Exception:
    # Inline rule-based insights
    churn_rate = clean_df['Churn'].mean() * 100
    print('=== RULE-BASED INSIGHTS ===')
    print()
    print(f'KEY FINDINGS:')
    print(f'  • Overall churn rate is {churn_rate:.1f}%')
    if 'Contract' in clean_df.columns:
        worst = clean_df.groupby('Contract')['Churn'].mean().idxmax()
        print(f'  • Highest churn contract type: {worst}')
    if 'tenure' in clean_df.columns:
        avg_t = clean_df[clean_df['Churn']==1]['tenure'].mean()
        print(f'  • Average tenure at churn: {avg_t:.1f} months')
    print()
    print('RECOMMENDATIONS:')
    print('  1. Offer discounts/incentives to Month-to-month contract customers to upgrade')
    print('  2. Prioritise retention calls for customers with tenure < 12 months')
    print('  3. Investigate quality issues for Fiber Optic service customers')
    print('  4. Introduce loyalty rewards for customers approaching the 24-month mark')
    print('  5. Create personalized retention packages for high-spend at-risk customers')

---
## Step 11: Save Outputs

In [ ]:
# ── Save predictions and model ────────────────────────────────────────────
import joblib

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save predictions CSV
pred_df.to_csv('../outputs/predictions.csv', index=False)
print('✓ Saved: outputs/predictions.csv')

# Save high-risk customers
if 'Risk Level' in pred_df.columns:
    high_risk_df = pred_df[pred_df['Risk Level'] == 'High Risk']
    high_risk_df.to_csv('../outputs/high_risk_customers.csv', index=False)
    print(f'✓ Saved: outputs/high_risk_customers.csv ({len(high_risk_df):,} customers)')

# Save best model
joblib.dump(best_model, '../models/churn_model.pkl')
print(f'✓ Saved: models/churn_model.pkl ({best_name})')

# Save preprocessor
joblib.dump(preprocessor, '../models/preprocessor.pkl')
print('✓ Saved: models/preprocessor.pkl')

---
## Summary

This notebook demonstrated a complete **AI-Powered Customer Churn Analysis** pipeline:

| Step | What Was Done |
|------|---------------|
| Data Loading | Loaded and audited raw customer data |
| Data Cleaning | Automated pipeline: deduplication, type coercion, null handling |
| EDA | 6+ interactive Plotly charts revealing churn patterns |
| Business KPIs | Calculated churn rate, revenue at risk, avg tenure |
| SQL Analysis | 4 business queries via Pandas |
| Preprocessing | StandardScaler + OneHotEncoder + stratified 80/20 split |
| Model Training | Logistic Regression, Random Forest, Gradient Boosting, XGBoost |
| Evaluation | Confusion Matrix, ROC Curve, Feature Importance |
| Predictions | Per-customer probability + Low/Medium/High risk tiers |
| AI Insights | Rule-based & LLM-enhanced findings + retention recommendations |

### Key Findings
- Month-to-month contracts have the highest churn rate
- Customers who churn have significantly shorter tenure
- High monthly charges correlate with increased churn probability
- ROC-AUC was used as the primary model selection criterion (more reliable than accuracy for imbalanced data)

### To run the interactive Streamlit dashboard:
```bash
streamlit run app.py
```

---
*Submitted by: **Sumit Gupta** | Project: AI-Powered Customer Churn Analysis*